In [1]:
import numpy as np
import numpy.random as rnd
import numpy.linalg as linalg
import math
from scipy.linalg import sqrtm
from scipy.optimize import minimize
from scipy.special import logsumexp
from numba import njit
from mpi4py import MPI

RuntimeError: cannot load MPI library
Could not find module 'C:\Users\Adam\AppData\Roaming\Python\DLLs' (or one of its dependencies). Try using the full path with constructor syntax.
Could not find module 'C:\Users\Adam\AppData\Roaming\Python\Library\bin' (or one of its dependencies). Try using the full path with constructor syntax.
Could not find module 'c:\Users\Adam\AppData\Local\Programs\Python\Python311\DLLs\impi.dll' (or one of its dependencies). Try using the full path with constructor syntax.
Could not find module 'c:\Users\Adam\AppData\Local\Programs\Python\Python311\DLLs\msmpi.dll' (or one of its dependencies). Try using the full path with constructor syntax.
Could not find module 'c:\Users\Adam\AppData\Local\Programs\Python\Python311\Library\bin' (or one of its dependencies). Try using the full path with constructor syntax.
Could not find module 'impi.dll' (or one of its dependencies). Try using the full path with constructor syntax.
Could not find module 'msmpi.dll' (or one

Functions for the State Evolution loop functions

In [4]:
def Softplus(x):
    return logsumexp([0, x])

def DSoftplus(x):
    return np.exp(-logsumexp([0, -x]))

def DDSoftplus(x):
    return np.exp(-x -2*logsumexp([0, -x]))

def Loss(y, z):
    return ((y - z[0])*(y - z[0])/(2*Softplus(z[1])) + np.log(Softplus(z[1])))/2

def DDzLoss(y, z):
    Sz = Softplus(z[1])
    Dsz = DSoftplus(z[1])
    return np.array([[1/Sz, (y - z[0])*Dsz/(Sz*Sz)],[(y - z[0])*Dsz/(Sz*Sz), (Sz*DDSoftplus(z[1])*(Sz - (y - z[0])*(y - z[0])) + Dsz*Dsz*(2*(y - z[0])*(y - z[0]) - Sz))/(2*Sz*Sz*Sz)]])

def Prox(mu, Omega, f):
    ToOptimize = lambda x: np.einsum("i,ij,j->",(x - mu),linalg.inv(Omega), (x - mu))/2 + f(x)
    Prox = minimize(ToOptimize, x0=[0, 0])
    return Prox.x

def T(mhat, qhat, chihat, W, xi):
    sqrtqhat = sqrtm(qhat).real
    return linalg.inv(chihat) @ (mhat @ W + sqrtqhat @ xi)

def fw(R, SigmaInv, Lambda):
    return linalg.inv(SigmaInv + Lambda*np.eye(2)) @ SigmaInv @ R

def fc(SigmaInv, Lambda):
    return linalg.inv(SigmaInv + Lambda*np.eye(2))

def phi(z, A):
    return z[0] + A*np.sqrt(Softplus(z[1]))

def Dz_phi(z, A):
    return np.array([1, np.divide(A*DSoftplus(z[1]), 2*np.sqrt(Softplus(z[1])))])

def gout(zStar, w, V):
    return linalg.inv(V) @ (zStar - w)

def Domega_gout(zStar, y, V):
    InvV = linalg.inv(V)
    return InvV @ (linalg.inv(InvV + DDzLoss(y, zStar)) @ InvV - np.eye(2))

def Dy_gout(zStar, y, V):
    InvV = linalg.inv(V)
    Sz = Softplus(zStar[1])
    Dsz = DSoftplus(zStar[1])
    return InvV @ (linalg.inv(InvV + DDzLoss(y, zStar)) @ np.array([1/Sz, (Dsz*(y - zStar[0]))/(Sz*Sz)]))

State Evolution Loop functions

In [5]:
def RealFuncs(mhat, qhat, chihat, W, xi, Lambda):
    fwval = fw(T(mhat, qhat, chihat, W, xi), chihat, Lambda)
    m = np.einsum("i,j->ij", fwval, W)
    q = np.einsum("i,j->ij", fwval, fwval)
    sigma = fc(chihat, Lambda)
    return m, q, sigma

def HatFuncs(sigma, z, w, A):
    y = phi(z, A)
    Optimizedz = Prox(w, sigma, lambda z: Loss(y, z))
    qhat = np.einsum("i,j->ij", gout(Optimizedz, w, sigma), gout(Optimizedz, w, sigma))
    mhat = np.einsum("i,j->ij", Dy_gout(Optimizedz, y, sigma), Dz_phi(z, A))
    chihat = Domega_gout(Optimizedz, y, sigma)
    return qhat, mhat, chihat

State Evolution sampling functions

In [6]:
def TrueRandSampleReal():
    W = rnd.normal(0, 1, 2)
    xi = rnd.normal(0, 1, 2)
    return W, xi

def TrueRandSampleHat(L):
    zw = L @ rnd.normal(0, 1, 4)
    A = rnd.normal(0, 1)
    return np.array([zw[0], zw[1]]), np.array([zw[2], zw[3]]), A

State Evolution Expectation functions

In [7]:
def TrueRandExpectReal(mhat, qhat, chihat, Lambda, Nsample):
    m, q, sigma = np.zeros((2, 2)), np.zeros((2, 2)), np.zeros((2, 2))
    for i in range(Nsample):
        W, xi = TrueRandSampleReal()
        newm, newq, newsigma = RealFuncs(mhat, qhat, chihat, W, xi, Lambda)
        m += newm
        q += newq
        sigma += newsigma
    return m/Nsample, q/Nsample, sigma/Nsample

def TrueRandExpectHat(q, m, sigma, alpha, Nsample):
    qhat, mhat, chihat = np.zeros((2, 2)), np.zeros((2, 2)), np.zeros((2, 2))
    L = linalg.cholesky(np.vstack([np.hstack([np.eye(2), m]), np.hstack([m, q])]))
    for i in range(Nsample):
        z, w, A = TrueRandSampleHat(L)
        newqhat, newmhat, newchihat = HatFuncs(sigma, z, w, A)
        qhat += newqhat
        mhat += newmhat
        chihat -= newchihat
    return alpha*qhat/Nsample, alpha*mhat/Nsample, alpha*chihat/Nsample

State Evolution runner

In [10]:
def TrueRandSE_ERM(alpha, Lambda, q0, m0, sigma0, Nsample = 1000, MaxIter = 1e4, EpsConvergence = 1e-6, Verbose = True, VerboseRate = 1, DebugVerbose = False):
    q, m, sigma = q0, m0, sigma0
    NIter = 0
    Conv = 1
    while((Conv > EpsConvergence) and (NIter < MaxIter)):
        qhat, mhat, chihat = TrueRandExpectHat(q, m, sigma, alpha, Nsample)
        newm, newq, newsigma = TrueRandExpectReal(mhat, qhat, chihat, Lambda, Nsample)
        Conv = (np.abs(q[0,0] - newq[0,0]) + np.abs(q[1,1] - newq[1,1]))/(np.abs(newq[0,0]) + np.abs(newq[1,1]))
        m, q, sigma = newm, newq, newsigma
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current convergence criterion %s" % Conv)
        if(DebugVerbose):
            print("qhat", qhat)
            print("mhat", mhat)
            print("chihat", chihat)
            print("m", m)
            print("q", q)
            print("sigma", sigma)
            print("Eigenvalues of Q", linalg.eigvals(np.vstack([np.hstack([np.eye(2), m]), np.hstack([m, q])])))
        NIter += 1
    return q, m, sigma

Main

In [ ]:
alpha = 10
Lambda = 1
q0 = np.array([[1, 0], [0, 0.81]])#0.5*np.eye(2)
m0 = np.array([[0.95, 0], [0, 0.75]])#0.2*np.eye(2)
sigma0 = 0.5*np.eye(2)
TrueRandSE_ERM(alpha, Lambda, q0, m0, sigma0, Nsample = 10000, DebugVerbose = True)

Iteration 0
Current convergence criterion 0.5836348602058844
qhat [[ 2.59294779 -0.03597451]
 [-0.03597451  0.96095801]]
mhat [[8.57617392 0.06274371]
 [0.0663897  0.91996843]]
chihat [[ 8.57617392  0.0663897 ]
 [ 0.0663897  -0.17501099]]
m [[ 0.91290088 -0.01467755]
 [ 0.00531008  1.20009878]]
q [[ 0.84080702 -0.00769802]
 [-0.00769802  2.7416592 ]]
sigma [[ 0.10448413 -0.0084082 ]
 [-0.0084082   1.21281399]]
Eigenvalues of Q [1.83676978 0.00426887 0.38784039 3.35358718]
Iteration 1
Current convergence criterion 1.0689731800550926
qhat [[9.0526612 0.0396338]
 [0.0396338 0.9401119]]
mhat [[15.47478354  0.42772577]
 [-0.25345751  1.29115999]]
chihat [[15.47478354 -0.25345751]
 [-0.25345751  0.75089273]]
m [[ 0.94321301  0.02787322]
 [-0.01395583  0.76754659]]
q [[0.92095935 0.03156726]
 [0.03156726 0.88803987]]
sigma [[0.06083431 0.00880631]
 [0.00880631 0.57241201]]
Eigenvalues of Q [0.01848013 0.17209063 1.90488567 1.7135428 ]
Iteration 2
Current convergence criterion 0.19383259201520

LinAlgError: Matrix is not positive definite